In [5]:
import os
import pandas as pd
import numpy as np
import json

# 1. 重新读取最原始的 CSV 文件
csv_path = '../data/scientists_raw.csv'  # 请确保路径和你的项目结构一致
df = pd.read_csv(csv_path)

# ==========================================================================
# 【前置步骤修复】在这里强制注入你定义的派生字段，确保它们百分之百存在！
# ==========================================================================

# 派生字段 1：本土/海外标记
df['is_diaspora'] = (df['cntry'] != 'grc').astype(int)

# 派生字段 2：学术年龄
df['academic_age'] = 2020 - df['firstyr']

# 派生字段 3：百分位分组
def percentile_group(p):
    if pd.isna(p):
        return 'unknown'
    if p <= 1:
        return 'top_1'
    elif p <= 5:
        return 'top_5'
    elif p <= 10:
        return 'top_10'
    else:
        return 'other'

# 【防错检查】检查原始数据里是不是叫 'top percentile'，确保列名完全一致
# 如果原始表里叫 'top percentile ' (有空格) 或别名字，请在这里修改
df['percentile_group'] = df['top percentile'].apply(percentile_group)

print('==== 前置派生字段检查 ====')
print(f"本土科学家数量: {(df['is_diaspora']==0).sum()}")
print(f"海外科学家数量: {(df['is_diaspora']==1).sum()}\n")

# ==========================================================================
# 2. 开始核心数据处理（计算各国影响力）
# ==========================================================================

influence_col = 'c (ns)'  # 排除自引的综合得分
df_clean = df.dropna(subset=[influence_col, 'cntry']).copy()

all_countries_stats = []

for country, group in df_clean.groupby('cntry'):
    scores = group[influence_col].sort_values()
    sample_size = len(group)
    
    q1 = scores.quantile(0.25)
    q3 = scores.quantile(0.75)
    iqr = q3 - q1
    
    country_stats = {
        "country": country,
        "count": int(sample_size),
        "mean": float(scores.mean()),
        "median": float(scores.median()),
        "min": float(scores.min()),
        "max": float(scores.max()),
        "q1": float(q1),
        "q3": float(q3),
        "lower_whisker": float(max(scores.min(), q1 - 1.5 * iqr)) if sample_size > 1 else float(scores.min()),
        "upper_whisker": float(min(scores.max(), q3 + 1.5 * iqr)) if sample_size > 1 else float(scores.max()),
        # 此时这里绝对不会再报 KeyError 了！
        "top_1_count": int((group['percentile_group'] == 'top_1').sum()),
        "top_5_count": int((group['percentile_group'] == 'top_5').sum()),
        "is_reliable": bool(sample_size >= 5) 
    }
    all_countries_stats.append(country_stats)

# 排序
all_countries_stats = sorted(all_countries_stats, key=lambda x: x['count'], reverse=True)

# 3. 宏观对比
macro_comparison = {}
for is_dias, group in df_clean.groupby('is_diaspora'):
    label = "diaspora" if is_dias == 1 else "local"
    scores = group[influence_col]
    macro_comparison[label] = {
        "count": int(len(group)),
        "mean": float(scores.mean()),
        "median": float(scores.median()),
        "top_1_pct": float((group['percentile_group'] == 'top_1').sum() / len(group) * 100)
    }

# 4. 导出 JSON
output_data = {
    "macro_summary": macro_comparison,
    "country_details": all_countries_stats
}

output_dir = '../web/data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'influence_all_countries.json')

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print("==== 导出状态 ====")
print(f"全量数据处理成功！共包含 {len(all_countries_stats)} 个国家。")
print(f"文件已保存在: {output_path}")

==== 前置派生字段检查 ====
本土科学家数量: 35116
海外科学家数量: 28835

==== 导出状态 ====
全量数据处理成功！共包含 108 个国家。
文件已保存在: ../web/data\influence_all_countries.json
